## imports and reads

In [1]:
from datetime import date
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

customers = spark.table("dbo_1.customers")
products  = spark.table("dbo_1.products")
orders    = spark.table("dbo_1.orders")
reviews   = spark.table("dbo_1.product_reviews")
social    = spark.table("dbo_1.social_media")
web_logs  = spark.table("dbo_1.web_logs")

StatementMeta(, 60ad814d-60ea-4d1c-9552-700fca4eec33, 3, Finished, Available, Finished, False)

## dimensions and fact table

In [2]:
dim_customer = customers.select("customerid", "customername", "email", "location", "signupdate")
dim_product  = products.select("productid", "productname", "category", "unitprice", "stock")
fact_orders  = orders.select("orderid", "orderdate", "customerid", "productid",
                             "quantity", "totalamount", "paymentmethod")

StatementMeta(, 60ad814d-60ea-4d1c-9552-700fca4eec33, 4, Finished, Available, Finished, False)

## date dimension

In [3]:
bounds = fact_orders.agg(F.min("orderdate").alias("first_day"),
                         F.max("orderdate").alias("last_day")).first()
start = date(bounds["first_day"].year, 1, 1)
end   = date(bounds["last_day"].year, 12, 31)

dim_date = (spark.range(1)
    .select(F.explode(F.sequence(F.lit(start), F.lit(end),
                                 F.expr("interval 1 day"))).alias("date"))
    .withColumn("year",         F.year("date"))
    .withColumn("quarter",      F.quarter("date"))
    .withColumn("month_number", F.month("date"))
    .withColumn("month_name",   F.date_format("date", "MMMM"))
    .withColumn("year_month",   F.date_format("date", "yyyy-MM"))
    .withColumn("day",          F.dayofmonth("date"))
    .withColumn("day_name",     F.date_format("date", "EEEE"))
    .withColumn("is_weekend",   F.dayofweek("date").isin(1, 7)))

StatementMeta(, 60ad814d-60ea-4d1c-9552-700fca4eec33, 5, Finished, Available, Finished, False)

## aggregates

In [4]:
agg_product_reviews = (reviews.groupBy("product_id")
    .agg(F.sum("rating").alias("rating_sum"), F.count("*").alias("review_count"))
    .withColumn("avg_rating", F.round(F.col("rating_sum") / F.col("review_count"), 2))
    .withColumnRenamed("product_id", "productid"))

agg_social_sentiment = (social
    .withColumn("event_date", F.to_date("event_ts"))
    .groupBy("event_date", "platform", "sentiment")
    .agg(F.count("*").alias("mention_count")))

agg_web_engagement = (web_logs
    .withColumn("event_date", F.to_date("event_ts"))
    .groupBy("event_date", "user_id", "page", "action")
    .agg(F.count("*").alias("action_count"))
    .withColumnRenamed("user_id", "customerid"))

StatementMeta(, 60ad814d-60ea-4d1c-9552-700fca4eec33, 6, Finished, Available, Finished, False)

## write to the dbo schema and reconcile

In [5]:
def save_table(df: DataFrame, name: str) -> None:
    (df.write.mode("overwrite").option("overwriteSchema", "true")
       .format("delta").saveAsTable(f"dbo.{name}"))
    print(f"saved dbo.{name}: {spark.table(f'dbo.{name}').count()} rows")

for df, name in [(dim_customer, "dim_customer"), (dim_product, "dim_product"),
                 (dim_date, "dim_date"), (fact_orders, "fact_orders"),
                 (agg_product_reviews,  "agg_product_reviews"),
                 (agg_social_sentiment, "agg_social_sentiment"),
                 (agg_web_engagement,   "agg_web_engagement")]:
    save_table(df, name)

silver_total = orders.agg(F.sum("totalamount")).first()[0]
gold_total   = spark.table("dbo.fact_orders").agg(F.sum("totalamount")).first()[0]
assert silver_total == gold_total, (silver_total, gold_total)
print("Reconciled total sales:", gold_total)

StatementMeta(, 60ad814d-60ea-4d1c-9552-700fca4eec33, 7, Finished, Available, Finished, False)

saved dbo.dim_customer: 100 rows
saved dbo.dim_product: 50 rows
saved dbo.dim_date: 1096 rows
saved dbo.fact_orders: 2939 rows
saved dbo.agg_product_reviews: 50 rows
saved dbo.agg_social_sentiment: 2712 rows
saved dbo.agg_web_engagement: 3000 rows
Reconciled total sales: 769420.82
